# Band Structure (HSE)

Calculate the electronic band structure of a material using the HSE hybrid functional with Quantum ESPRESSO on the Mat3ra platform.

> **Note:** `SCF_KGRID` must be set to a uniform Gamma-centered grid. QE requires the exchange q-grid (`nqx1/2/3`) to match the k-mesh; a mismatch causes a `k + q is not an S*k` error.

<h2 style="color:green">Usage</h2>

1. Set material and calculation parameters in cells 1.2 and 1.3 below. `SCF_KGRID` in cell 1.3 is required.
1. Click "Run" > "Run All" to run all cells.
1. Wait for the job to complete.
1. Scroll down to view the result.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure parameters for material, workflow, compute resources, and job.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Create material: materials are read from the `../uploads` folder — place files there manually or run a material creation notebook first. If the material is not found by name, Standata is used as a fallback. The material is then saved to the platform.
1. Configure workflow: load the HSE band structure workflow from Standata, set model, set k-grid (required) and energy cutoffs, and save the workflow.
1. Configure compute: get list of clusters and create compute configuration with selected cluster, queue, and number of processors.
1. Create the job with material and workflow configuration: assemble the job from material, workflow, project, and compute configuration.
1. Submit the job and monitor the status: submit the job and wait for completion.
1. Retrieve results: get and display the band structure.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
import sys

if sys.platform == "emscripten":
    import micropip

    await micropip.install('mat3ra-notebooks-utils')
    from mat3ra.notebooks_utils.jupyterlite.packages import install_packages

    await install_packages("api_examples")

### 1.2. Set parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
MATERIAL_NAME = "Silicon"

# 4. Workflow parameters
CALCULATION_TYPE = "band_structure_hse"

APPLICATION_NAME = "espresso"
ADD_RELAXATION = False

WORKFLOW_SEARCH_TERM = f"{CALCULATION_TYPE}.json"
MY_WORKFLOW_NAME = "Band Structure HSE" + (" (relax)" if ADD_RELAXATION else "")

# Model parameters
MODEL_SUBTYPE = "hybrid"  # HSE hybrid functional

# 5. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds

### 1.3. Set specific HSE band structure parameters

In [ ]:
# Method parameters
PSEUDOPOTENTIAL_TYPE = "us"   # ultrasoft pseudopotentials (standard for HSE in QE)
FUNCTIONAL = "hse06"

# K-grids and k-path
RELAXATION_KGRID = None  # e.g. [4, 4, 4]
# Required: uniform Gamma-centered grid — sets both k-mesh and exchange q-grid (nqx1/2/3)
SCF_KGRID = [1, 1, 1]
KPATH = None              # e.g. [{"point": "G", "steps": 20}, {"point": "X", "steps": 20}]

# Energy cutoffs
ECUTWFC = 40
ECUTRHO = 200

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable "OIDC_ACCESS_TOKEN".

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"\u2705 Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"\u2705 Using project: {projects[0]['name']} ({project_id})")

## 3. Create material
### 3.1. Load material from local file (or Standata)

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.python.other.material.visualize import visualize_materials as visualize
from mat3ra.notebooks_utils.material import load_material_from_folder

material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME))

visualize(material)

### 3.2. Save material to the platform

In [ ]:
from mat3ra.notebooks_utils.python.other.api.helpers import get_or_create_material

material.basis.set_labels_from_list([])
saved_material_response = get_or_create_material(client, material, ACCOUNT_ID)
saved_material = Material.create(saved_material_response)

## 4. Configure workflow
### 4.1. Select application

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.ade.application import Application

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

### 4.2. Load workflow from Standata and preview it

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.python.other.workflow.visualize import visualize_workflow

workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(WORKFLOW_SEARCH_TERM)
workflow = Workflow.create(workflow_config)
workflow.name = MY_WORKFLOW_NAME

visualize_workflow(workflow)

### 4.3. Add relaxation (optional)

In [ ]:
if ADD_RELAXATION:
    workflow.add_relaxation()
    visualize_workflow(workflow)

### 4.4. Set Model and its parameters (physics)

In [ ]:
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.mode.model import Model

model_config = ModelTreeStandata.get_model_by_parameters(
    type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL
)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = Model.create(model_config)

preliminary_scf_subworkflow = next(
    swf for swf in workflow.subworkflows if swf.name == "Preliminary SCF Calculation"
)
main_hse_subworkflow = next(
    swf for swf in workflow.subworkflows if swf.name == "Main HSE Run"
)
main_hse_subworkflow.model = model

visualize_workflow(workflow)

### 4.5. Modify Method (computational parameters): k-grid, k-path, and cutoffs

`SCF_KGRID` sets both the k-mesh and the exchange q-grid (`nqx1/2/3`). They must match for HSE to succeed.

In [ ]:
from mat3ra.wode.context.providers import PlanewaveCutoffsContextProvider, PointsGridDataProvider, \
    PointsPathDataProvider

preliminary_scf_subworkflow = next(
    swf for swf in workflow.subworkflows if swf.name == "Preliminary SCF Calculation"
)
main_hse_subworkflow = next(
    swf for swf in workflow.subworkflows if swf.name == "Main HSE Run"
)

if RELAXATION_KGRID is not None and ADD_RELAXATION:
    unit = workflow.subworkflows[0].get_unit_by_name(name_regex="relax")
    unit.add_context(PointsGridDataProvider(dimensions=RELAXATION_KGRID, isEdited=True).yield_data())
    workflow.subworkflows[0].set_unit(unit)

if SCF_KGRID is not None:
    unit = preliminary_scf_subworkflow.get_unit_by_name(name="pw_scf")
    if unit:
        unit.add_context(PointsGridDataProvider(dimensions=SCF_KGRID, isEdited=True).yield_data())
        preliminary_scf_subworkflow.set_unit(unit)

    unit = main_hse_subworkflow.get_unit_by_name(name="pw_scf_bands_hse")
    if unit:
        unit.add_context(PointsGridDataProvider(dimensions=SCF_KGRID, isEdited=True).yield_data())
        unit.add_context(PointsGridDataProvider(name="qgrid", dimensions=SCF_KGRID, isEdited=True).yield_data())
        main_hse_subworkflow.set_unit(unit)

if KPATH is not None:
    unit = main_hse_subworkflow.get_unit_by_name(name="pw_scf_bands_hse")
    if unit:
        unit.add_context(PointsPathDataProvider(path=KPATH, isEdited=True).yield_data())
        main_hse_subworkflow.set_unit(unit)

if ECUTWFC is not None:
    cutoffs_context = PlanewaveCutoffsContextProvider(wavefunction=ECUTWFC, density=ECUTRHO, isEdited=True).yield_data()
    for unit_name in ["pw_relax", "pw_vc-relax", "pw_scf_bands_hse"]:
        for swf in workflow.subworkflows:
            unit = swf.get_unit_by_name(name=unit_name)
            if unit:
                unit.add_context(cutoffs_context)
                swf.set_unit(unit)

### 4.6. Preview final workflow

In [ ]:
visualize_workflow(workflow)
workflow.to_dict()

### 4.7. Save workflow to collection

In [ ]:
from mat3ra.notebooks_utils.python.other.api.helpers import get_or_create_workflow

saved_workflow_response = get_or_create_workflow(client, workflow, ACCOUNT_ID)
saved_workflow = Workflow.create(saved_workflow_response)
print(f"Workflow ID: {saved_workflow.id}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create compute configuration

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(
    cluster=cluster,
    queue=QUEUE_NAME,
    ppn=PPN
)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create the job
### 6.1. Create job

In [ ]:
from mat3ra.notebooks_utils.python.other.api.helpers import create_job
from mat3ra.utils.object import dict_to_namespace_recursive
from mat3ra.notebooks_utils import display_JSON

print(f"Material: {saved_material.id}")
print(f"Workflow: {saved_workflow.id}")
print(f"Project: {project_id}")

job_name = MY_WORKFLOW_NAME + " " + saved_material.formula + " " + timestamp
job_response = create_job(
    api_client=client,
    materials=[saved_material],
    workflow=workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=job_name,
    compute=compute.to_dict(),
)

job = dict_to_namespace_recursive(job_response)
job_id = job._id
print("\u2705 Job created successfully!")
print(f"Job ID: {job_id}")
display_JSON(job_response)

## 7. Submit the job and monitor the status

In [ ]:
client.jobs.submit(job_id)
print(f"\u2705 Job {job_id} submitted successfully!")

In [ ]:
from mat3ra.notebooks_utils.python.other.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)

## 8. Retrieve and visualize results
### 8.1. Band Structure

In [ ]:
from mat3ra.prode import PropertyName
from mat3ra.notebooks_utils.python.other.api.helpers import get_properties_for_job
from mat3ra.notebooks_utils.python.other.property.visualize import visualize_properties

band_structure_data = get_properties_for_job(client, job_id)
visualize_properties(band_structure_data, title="Band Structure (HSE)", extra_config={"material":material.to_dict()})